# Localizer × Typing 비교 실험 (B0 / B1 / B2)

**질문:** "결함 위치를 먼저 추리고 그 영역 위에서만 종류/others를 판정"하면 전역 pooling 대비 나아지는가?
그리고 **학습 재구성 localizer(Dinomaly-lite)** 가 단순 kNN 메모리를 이기는가?

| | Stage1 localizer | Stage2 결함영역 → descriptor | Stage3 |
|---|---|---|---|
| **B0 (현행)** | 정상 coreset kNN anomaly | **고정 top-p% pooling** | logreg + descriptor OOD |
| **B1** | 정상 coreset kNN anomaly | **정상보정 임계로 영역 추출** | logreg + descriptor OOD |
| **B2** | **학습 재구성**(transformer 디코더 + spatial attention + noisy bottleneck) | 정상보정 영역 추출 | logreg + descriptor OOD |

- **B0 vs B1** : 같은 localizer, descriptor만 다름 → **영역 명시화 효과** 격리.
- **B1 vs B2** : 같은 영역·종류 하니스, localizer만 다름 → **학습 재구성 값어치** 격리.
- 탐지 AUROC는 localizer만의 함수 → B0==B1(같은 kNN), B2만 다름.

**핵심 지표 = leave-one-class-out open-set AUROC** (한 불량을 학습에서 빼고 others로 잡는 능력, threshold-free).

B2 재구성 localizer = frozen DINO patch feature를 정상만으로 재구성하도록 얕은 transformer를 학습,
`anomaly = 1 − cosine(재구성, 원본)`. noisy bottleneck으로 이상까지 복원하는 identity-shortcut 억제.

> 서버는 torch+GPU 환경. B0·B1은 torch-free, **B2만 torch** 사용. feature는 feature service(`?include=patch`).



In [ ]:
import os, sys, hashlib
from pathlib import Path
import numpy as np

# --- repo import (feature service 클라이언트) ---
REPO = Path.cwd()
while REPO != REPO.parent and not (REPO / "inspection").is_dir():
    REPO = REPO.parent
for p in (str(REPO), str(REPO / "dino_v3")):
    if p not in sys.path:
        sys.path.insert(0, p)
from inspection.service.client_example import get_patch_features, _list_images
from inspection.em_classifier import ClassifierHead   # torch-free 부분

# ============================ 설정 ============================
SERVER     = os.environ.get("EM_SERVER", "http://localhost:8000")
DATA_ROOT  = os.environ.get("EM_DATA_ROOT", "/data/classes")   # <class>/*.png, 정상=normal
NORMAL     = "normal"
SEED       = 0
SPLIT      = 0.5      # support 비율 (나머지 query)
K_NN       = 3        # kNN localizer
CORESET    = 2000     # 정상 메모리 coreset 크기
TOPP       = 1.0      # 이미지 anomaly = 상위 topp% patch 평균 (탐지 공통)
REGION_PCT = 99.0     # B1/B2 영역 임계 = 정상 patch anomaly 분포의 이 퍼센타일
NORMAL_FIT_CAP = 20000  # localizer 적합에 쓸 정상 patch 상한
CACHE      = Path(os.environ.get("EM_CACHE", "./_patchcache"))  # 이미지별 patch feature 캐시

print("SERVER   =", SERVER)
print("DATA_ROOT=", DATA_ROOT)
print("classes  =", sorted([d.name for d in Path(DATA_ROOT).iterdir() if d.is_dir()]) if Path(DATA_ROOT).is_dir() else "(경로 없음)")


## 1. 데이터 분할 + patch feature 캐시

클래스별로 support/query 를 고정 시드로 나눈다. **memory·logreg·임계는 support 로만**,
모든 지표는 **query 로만** — 이전 실험의 test 오염 문제를 원천 차단.

patch feature 는 이미지당 한 번만 서버에서 받아 디스크 캐시(`_patchcache/`). 백본이 바뀌면 캐시를 지울 것.


In [ ]:
def _key(p):
    return hashlib.sha1(str(Path(p).resolve()).encode()).hexdigest()[:16]

def fetch_patch(paths, batch=8):
    CACHE.mkdir(parents=True, exist_ok=True)
    todo = [p for p in paths if not (CACHE / f"{_key(p)}.npz").exists()]
    for i in range(0, len(todo), batch):
        chunk = todo[i:i + batch]
        got = get_patch_features(SERVER, chunk, batch=batch)   # 순서 보존
        for p, (pf, h, w, _name) in zip(chunk, got):
            np.savez(CACHE / f"{_key(p)}.npz", pf=pf.astype(np.float32), hw=np.array([h, w]))
        print(f"  fetched {min(i+batch,len(todo))}/{len(todo)}", end="\r")
    out = {}
    for p in paths:
        d = np.load(CACHE / f"{_key(p)}.npz")
        out[p] = (d["pf"], int(d["hw"][0]), int(d["hw"][1]))
    return out

def split_images(root, split, seed):
    rng = np.random.default_rng(seed)
    sup, qry = {}, {}
    for d in sorted(Path(root).iterdir()):
        if not d.is_dir():
            continue
        imgs = _list_images(str(d)); rng.shuffle(imgs)
        k = max(1, int(len(imgs) * split))
        sup[d.name], qry[d.name] = imgs[:k], imgs[k:]
    return sup, qry

def build_feats(split_dict):
    allp = [p for c in split_dict for p in split_dict[c]]
    cache = fetch_patch(allp)
    return {c: [cache[p] for p in split_dict[c]] for c in split_dict}

sup_paths, qry_paths = split_images(DATA_ROOT, SPLIT, SEED)
for c in sup_paths:
    print(f"{c:16s} support {len(sup_paths[c]):4d} / query {len(qry_paths[c]):4d}")

sup = build_feats(sup_paths)
qry = build_feats(qry_paths)
defect_names = [c for c in sorted(sup) if c != NORMAL]
print("\n정상=", NORMAL, " 불량=", defect_names)


## 2. Stage 1 — localizer (patch anomaly map)

정상 support 만으로 "정상이 어떻게 생겼나"를 세우고 patch별 이상 점수 `a[N]` 를 낸다.
- **kNN** (B0/B1): coreset 메모리와의 top-k cosine → `1 − sim` (PatchCore 계열, 무학습).
- **recon** (B2): 정상 feature를 재구성하는 얕은 transformer를 학습, `1 − cosine(재구성, 원본)` (§6b).

모든 localizer 는 **이미지별 patch 배열 리스트**를 받는다(재구성은 토큰 간 attention 때문에 이미지 경계 필요).
두 점수의 스케일은 달라도 각 파이프라인이 자기 임계를 support 로 보정하므로 무방.


In [ ]:
def l2(x):
    x = np.asarray(x, np.float32)
    return x / (np.linalg.norm(x, axis=-1, keepdims=True) + 1e-12)

def coreset(feats, size, cap=12000, seed=0):
    feats = l2(feats); n = len(feats)
    if n <= size:
        return feats
    r = np.random.default_rng(seed)
    if n > cap:
        feats = feats[r.choice(n, cap, replace=False)]; n = len(feats)
    sel = [int(r.integers(n))]; mx = feats @ feats[sel[0]]
    for _ in range(size - 1):
        i = int(np.argmin(mx)); sel.append(i); mx = np.maximum(mx, feats @ feats[i])
    return feats[np.array(sel)]

class KNNLocalizer:
    name = "kNN"
    def __init__(self, normal_imgs, k=K_NN, size=CORESET):
        self.M = coreset(np.vstack(normal_imgs), size); self.k = k
    def score(self, pf):
        pf = l2(pf); out = np.empty(len(pf), np.float32)
        for i in range(0, len(pf), 4096):
            s = pf[i:i+4096] @ self.M.T; k = min(self.k, s.shape[1])
            out[i:i+4096] = 1.0 - np.partition(s, -k, axis=1)[:, -k:].mean(1)
        return out


## 3. Stage 2·3 — 영역 추출 → descriptor → 종류/OOD

- **B0 (top-p%)** : 항상 상위 top-p% patch 평균 → 정상 이미지도 "가장 덜 정상인" patch로 descriptor가 생겨 잡음.
- **B1/B2 (region)** : 정상 patch anomaly 분포의 `REGION_PCT` 퍼센타일을 임계로, 그걸 넘는 patch만 평균.
  결함이 크면 많이, 없으면 거의 비어 최이상 patch 1개로 폴백 → descriptor가 결함에 집중.

descriptor 위에 logreg(종류) + 학습 descriptor와의 최대 cosine(OOD=others 근거).


In [ ]:
def image_score(a, topp=TOPP):                 # 탐지용 이미지 점수(공통)
    k = max(1, int(len(a) * topp / 100))
    return float(np.sort(a)[-k:].mean())

def desc_topp(pf, a, topp=TOPP):               # B0
    k = max(1, int(len(a) * topp / 100))
    return pf[np.argsort(a)[-k:]].mean(0)

def desc_region(pf, a, thr, min_k=1):          # B1/B2
    idx = np.where(a > thr)[0]
    if len(idx) < min_k:
        idx = np.argsort(a)[-min_k:]
    return pf[idx].mean(0)

class Pipeline:
    def __init__(self, name, localizer, mode):
        self.name = name; self.loc = localizer; self.mode = mode
        self.region_thr = None
    def ia_desc(self, pf):
        a = self.loc.score(pf)
        ia = image_score(a)
        d = desc_topp(pf, a) if self.mode == "topp" else desc_region(pf, a, self.region_thr)
        return ia, d, a

def build(name, localizer_cls, mode, sup_feats, defect_names):
    normal_imgs = [f for f, _, _ in sup_feats[NORMAL]]
    pipe = Pipeline(name, localizer_cls(normal_imgs), mode)
    # 영역 임계: 정상 support patch anomaly 분포
    ns = np.concatenate([pipe.loc.score(f) for f in normal_imgs])
    pipe.region_thr = float(np.percentile(ns, REGION_PCT))
    # 종류 logreg : 불량 descriptor
    X, y = [], []
    for ci, c in enumerate(defect_names):
        for f, _, _ in sup_feats[c]:
            X.append(pipe.ia_desc(f)[1]); y.append(ci)
    X = np.vstack(X); y = np.array(y)
    pipe.head = ClassifierHead.fit(X, y, defect_names, kind="logreg")
    pipe.desc_mem = l2(X); pipe.defect_names = defect_names
    # 탐지 임계 : Youden (정상 vs 불량 이미지)
    sc, lb = [], []
    for c in sup_feats:
        for f, _, _ in sup_feats[c]:
            sc.append(pipe.ia_desc(f)[0]); lb.append(0 if c == NORMAL else 1)
    sc = np.array(sc); lb = np.array(lb); ts = np.unique(sc)
    # 임계 후보 = 관측 점수 사이 중점. 분리 구간이면 정상 최댓값에 붙지 않고 gap 중앙을
    # 고르게 되어 신규 정상(query)에 강건(정상 max 에 붙으면 신규 정상 과탐).
    cand = (ts[:-1] + ts[1:]) / 2 if len(ts) > 1 else ts
    jv = [np.mean(sc[lb == 1] > t) - np.mean(sc[lb == 0] > t) for t in cand]
    pipe.det_thr = float(cand[int(np.argmax(jv))])
    return pipe


## 4. 지표 — 탐지 / 종류(closed-set)

- **탐지 AUROC** : 이미지 anomaly 로 정상 vs 불량 분리(threshold-free).
- **종류 정확도** : 실제 불량 query 에 대한 descriptor→logreg argmax 정확도.
- **end-to-end 정확도** : 게이트(정상/불량) 후 종류까지, normal 포함 전체.


In [ ]:
from sklearn.metrics import roc_auc_score

def eval_query(pipe, qry_feats):
    rows = []
    for c in qry_feats:
        for f, h, w in qry_feats[c]:
            ia, d, _ = pipe.ia_desc(f)
            proba = pipe.head.predict_proba(d[None])[0]
            ood = float((l2(d) @ pipe.desc_mem.T).max())
            rows.append(dict(cls=c, is_def=int(c != NORMAL), ia=ia,
                             pred=pipe.defect_names[int(proba.argmax())],
                             maxp=float(proba.max()), ood=ood))
    return rows

def closed_metrics(pipe, rows):
    y = np.array([r["is_def"] for r in rows]); s = np.array([r["ia"] for r in rows])
    det_auc = float(roc_auc_score(y, s)) if len(set(y)) > 1 else float("nan")
    dz = [r for r in rows if r["is_def"] == 1]
    typ = float(np.mean([r["pred"] == r["cls"] for r in dz])) if dz else float("nan")
    e2e = float(np.mean([((NORMAL if r["ia"] < pipe.det_thr else r["pred"]) == r["cls"]) for r in rows]))
    return det_auc, typ, e2e


## 5. 지표 — open-set others (leave-one-class-out) ← 핵심

불량 클래스를 하나씩 학습에서 빼고(=미지), 그 클래스 query 가 **others 로 밀려나는가**를 본다.
OOD 점수(=학습 descriptor와의 최대 cosine)로 known-defect query vs held-out query 를 가르는
**AUROC**(threshold-free). 1에 가까울수록 others 를 잘 잡는 것.


In [ ]:
def eval_openset(localizer_cls, mode, sup_feats, qry_feats, defect_names):
    aucs = {}
    for held in defect_names:
        known = [c for c in defect_names if c != held]
        if not known:
            continue
        sub = {c: sup_feats[c] for c in [NORMAL] + known}
        pipe = build("loo", localizer_cls, mode, sub, known)
        known_ood = [float((l2(pipe.ia_desc(f)[1]) @ pipe.desc_mem.T).max())
                     for c in known for f, _, _ in qry_feats[c]]
        held_ood  = [float((l2(pipe.ia_desc(f)[1]) @ pipe.desc_mem.T).max())
                     for f, _, _ in qry_feats[held]]
        y = np.r_[np.zeros(len(known_ood)), np.ones(len(held_ood))]
        s = np.r_[-np.array(known_ood), -np.array(held_ood)]   # 멀수록(others) 점수 높게
        aucs[held] = float(roc_auc_score(y, s)) if len(set(y)) > 1 and len(held_ood) else float("nan")
    return aucs


## 6. 실행 — B0 / B1 (kNN, torch-free)


In [ ]:
built, summary = {}, []

def run_pipe(name, lc, mode):
    pipe = build(name, lc, mode, sup, defect_names); built[name] = pipe
    det_auc, typ, e2e = closed_metrics(pipe, eval_query(pipe, qry))
    os_auc = eval_openset(lc, mode, sup, qry, defect_names)
    os_mean = float(np.nanmean(list(os_auc.values()))) if os_auc else float("nan")
    summary.append(dict(pipe=name, localizer=pipe.loc.name, region=mode,
                        det_auc=round(det_auc, 3), typing_acc=round(typ, 3),
                        e2e_acc=round(e2e, 3), openset_auc=round(os_mean, 3),
                        **{f"os[{k}]": round(v, 3) for k, v in os_auc.items()}))
    print(f"[{name}] det={det_auc:.3f} typing={typ:.3f} e2e={e2e:.3f} openset={os_mean:.3f}  {os_auc}")

run_pipe("B0", KNNLocalizer, "topp")
run_pipe("B1", KNNLocalizer, "region")


## 6b. B2 — 학습 재구성 localizer (Dinomaly-lite, torch)

frozen DINO patch feature 를 정상만으로 재구성하는 얕은 **transformer 디코더**(토큰 간 self-attention =
spatial 문맥). bottleneck 에 노이즈를 주입해 이상까지 복원하는 identity-shortcut 을 억제.
`anomaly = 1 − cosine(재구성, 원본)`. B1 과 **같은 region·logreg·OOD** 하니스 재사용.

> torch 필요(서버 GPU). open-set 평가는 held-out 클래스마다 재학습(수십 초/회).


In [ ]:
try:
    import torch
    import torch.nn as nn

    class _ReconNet(nn.Module):
        def __init__(self, D, m, layers, heads, drop):
            super().__init__()
            self.inp = nn.Linear(D, m)
            enc = nn.TransformerEncoderLayer(m, heads, 4 * m, dropout=drop,
                                             batch_first=True, activation="gelu")
            self.dec = nn.TransformerEncoder(enc, layers)
            self.out = nn.Linear(m, D)
        def forward(self, x, noise=0.0):
            z = self.inp(x)
            if noise > 0:
                z = z + torch.randn_like(z) * noise      # noisy bottleneck
            r = self.out(self.dec(z))
            return r / (r.norm(dim=-1, keepdim=True) + 1e-8)

    class ReconLocalizer:
        name = "recon"
        def __init__(self, normal_imgs, m=256, layers=2, heads=4, epochs=60, lr=1e-3,
                     noise=0.3, drop=0.1, cap_imgs=400, seed=0, device=None, verbose=False):
            D = normal_imgs[0].shape[1]
            self.dev = device or ("cuda" if torch.cuda.is_available() else "cpu")
            torch.manual_seed(seed)
            self.net = _ReconNet(D, m, layers, heads, drop).to(self.dev)
            imgs = normal_imgs
            if cap_imgs and len(imgs) > cap_imgs:
                r = np.random.default_rng(seed)
                imgs = [imgs[i] for i in r.choice(len(imgs), cap_imgs, replace=False)]
            groups = {}                                   # 토큰 수(N)별로 묶어 배치화
            for a in imgs:
                groups.setdefault(a.shape[0], []).append(l2(a))
            batches = [torch.tensor(np.stack(v), dtype=torch.float32) for v in groups.values()]
            opt = torch.optim.Adam(self.net.parameters(), lr=lr, weight_decay=1e-5)
            self.net.train()
            for ep in range(epochs):
                tot = nt = 0
                for b in batches:
                    xb = b.to(self.dev)
                    r = self.net(xb, noise=noise)
                    loss = (1.0 - (r * xb).sum(-1)).mean()
                    opt.zero_grad(); loss.backward(); opt.step()
                    tot += loss.item() * xb.shape[0] * xb.shape[1]; nt += xb.shape[0] * xb.shape[1]
                if verbose and (ep + 1) % 20 == 0:
                    print(f"  recon ep{ep+1}/{epochs} loss={tot/nt:.4f}")
            self.net.eval()
        def score(self, pf):
            x = torch.tensor(l2(pf)[None], dtype=torch.float32, device=self.dev)
            with torch.no_grad():
                r = self.net(x)                           # eval: noise=0
                return (1.0 - (r * x).sum(-1))[0].cpu().numpy().astype(np.float32)

    run_pipe("B2", ReconLocalizer, "region")
    try:
        import pandas as pd
        display(pd.DataFrame(summary).set_index("pipe"))
    except Exception:
        [print(r) for r in summary]
except ImportError:
    print("torch 없음 → 재구성 localizer(B2) 건너뜀 (torch+GPU 서버에서 실행)")


**읽는 법**
- `B0 → B1` typing/openset ↑ → 영역 명시화가 값어치(전역 pooling 약점 확인).
- `B1 → B2` typing/openset ↑ → **학습 재구성 localizer가 kNN을 이김** → defect_model localizer 교체.
- `B2 ≈ B1` → kNN으로 충분(값싼 쪽 유지), 재구성 도입 보류.
- `det_auc` 는 B0==B1 이어야 정상(같은 kNN). 크게 다르면 버그 신호.
- openset 이 전부 낮으면 → OOD(others)를 거리-reject 말고 **축적 후 K+1 학습**으로 가야 한다는 근거.


## 7. localizer 육안 비교 (anomaly heatmap)

query 몇 장에 대해 원본 + 각 localizer map + B1 선택 영역을 본다. 숫자로 안 잡히는
localization 품질 차이(경계 선명도, 배경 오탐)를 확인.


In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

def show_localizers(n_per_class=2):
    provs = [("kNN", built["B0"].loc)]
    if "B2" in built:
        provs.append(("recon", built["B2"].loc))
    ncol = 1 + len(provs) + 1
    thr = built["B1"].region_thr
    rows_ = [(c, p) for c in qry_paths for p in qry_paths[c][:n_per_class]]
    fig, ax = plt.subplots(len(rows_), ncol, figsize=(3 * ncol, 3 * len(rows_)))
    ax = np.atleast_2d(ax)
    for i, (c, path) in enumerate(rows_):
        pf, h, w = qry[c][qry_paths[c].index(path)]
        img = np.array(Image.open(path).convert("L").resize((160, 160)))
        ax[i, 0].imshow(img, cmap="gray"); ax[i, 0].set_title(c); ax[i, 0].axis("off")
        for j, (nm, loc) in enumerate(provs, start=1):
            ax[i, j].imshow(loc.score(pf).reshape(h, w), cmap="inferno")
            ax[i, j].set_title(nm); ax[i, j].axis("off")
        mask = (built["B1"].loc.score(pf).reshape(h, w) > thr)
        ax[i, -1].imshow(img, cmap="gray")
        ax[i, -1].imshow(mask, cmap="Reds", alpha=0.4, extent=(0, 160, 160, 0))
        ax[i, -1].set_title("B1 region"); ax[i, -1].axis("off")
    plt.tight_layout(); plt.show()

show_localizers()


## 8. 결론 메모

실행 후 채울 것:
- B0 vs B1 vs B2 의 typing / **openset** 우열 →
- 영역 명시화가 값어치 있었나(B0→B1) →
- 학습 재구성 localizer가 kNN을 이겼나(B1→B2) →
- others 전략: 거리-reject 유지 vs 축적 후 K+1 학습 전환 →

이 결과로 `emclf_ui/defect_model.py` 의 localizer·descriptor·others 로직을 확정한다.
